In [1]:
import numpy as np
from astropy.table import Table
import astropy.units as u
from astropy.coordinates import SkyCoord

# 例：refine_events のFITS（calc_eventsのroot_events.fitsでもOK）
tab = Table.read("root_refine_events_I_cardelli89.fits")  # ファイル名はあなたの設定に合わせて

# 例：i番目のイベントを取り出す（条件で絞るならtab[mask][0]みたいに）
i = 0
row = tab[i]

# --- 1) 質量 ---
ML = row["mass_L"]  # Msun

# --- 2) 距離（kpc） ---
DL = np.sqrt(row["px_L"]**2 + row["py_L"]**2 + row["pz_L"]**2)  # kpc
DS = np.sqrt(row["px_S"]**2 + row["py_S"]**2 + row["pz_S"]**2)  # kpc

# --- 3) 固有運動：銀河(l,b) -> 赤道(ra,dec)に変換 ---
# PopSyCLEのmu_lcosb, mu_b は mas/yr で、glon/glat は deg（イベント表に_L,_Sで入ってる）
cL_gal = SkyCoord(
    l=row["glon_L"] * u.deg,
    b=row["glat_L"] * u.deg,
    distance=DL * u.kpc,
    pm_l_cosb=row["mu_lcosb_L"] * (u.mas/u.yr),
    pm_b=row["mu_b_L"] * (u.mas/u.yr),
    frame="galactic",
)
cS_gal = SkyCoord(
    l=row["glon_S"] * u.deg,
    b=row["glat_S"] * u.deg,
    distance=DS * u.kpc,
    pm_l_cosb=row["mu_lcosb_S"] * (u.mas/u.yr),
    pm_b=row["mu_b_S"] * (u.mas/u.yr),
    frame="galactic",
)

cL_icrs = cL_gal.icrs
cS_icrs = cS_gal.icrs

# ここは定義が2通りある（lens-source か source-lens か）
# microlensingでよく使う「レンズの相対運動」なら lens - source を採用することが多い
mu_rel_E = (cL_icrs.pm_ra_cosdec - cS_icrs.pm_ra_cosdec).to(u.mas/u.yr)  # East
mu_rel_N = (cL_icrs.pm_dec - cS_icrs.pm_dec).to(u.mas/u.yr)              # North

print("ML [Msun] =", ML)
print("DL [kpc]  =", DL, " DS [kpc] =", DS)
print("mu_rel_E  =", mu_rel_E, " mu_rel_N =", mu_rel_N)


FileNotFoundError: [Errno 2] No such file or directory: 'root_refine_events_I_cardelli89.fits'